In [1]:
from langchain_ollama import ChatOllama

# Step 2: Connect using the direct local Ollama channel
Model = ChatOllama(
    model="gemma4:e4b",
    base_url="http://127.0.0.1:11434", # Notice: NO '/v1' path suffix needed here
    temperature=1.0  ,                  # Recommended baseline sampling for Gemma 4
    num_ctx = 16384
)


In [2]:
from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults
#from langchain_tavily import TavilySearch
from deepagents import create_deep_agent
import os

In [5]:
# 1. Provide Domain-Specific Tools
# The agent will automatically blend these with its native file & planning tools.
web_search_tool = TavilySearchResults(
    max_results=5, 
    description="Useful for finding up-to-date market, competitor, and industry data."
)
custom_tools = [web_search_tool]


C:\Users\maxim\AppData\Local\Temp\ipykernel_11416\1576097492.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(


In [6]:

# 2. Define the Specialized Sub-Agents
# Sub-agents spawn dynamically inside isolated context windows to keep the main chat clean.
market_analyst_subagent = {
    "name": "market_analyst",
    "description": "Quant-driven analyst for Indian equities, market sizing, and sector trends.",
    "system_prompt": """You are an elite Indian Share Market Analyst specializing in SEBI-aligned technical and fundamental assessments.

CRITICAL OPERATIONAL RULES:
1. DATA SCOPE: Analyze NSE/BSE tickers, sector indices, market cap metrics, CAGR, and growth runways.
2. DISK PERSISTENCE: Write all complex numerical computations, data tables, and intermediate analyses directly to files. Do NOT dump raw data chunks into the chat context window.
3. CONTEXT EFFICIENCY: Keep your conversational text response under 300 words. Summarize key insights in the chat; point to the generated files for deep-dive datasets.

OUTPUT TEMPLATE (Strictly adhere to this format for chat responses):
- **Executive Summary**: 2-sentence macro overview.
- **Key Metrics Table**: Markdown table containing (Metric | Value | Impact).
- **Files Generated**: Bulleted list of absolute file paths containing intermediate data.
- **Strategic Verdict**: Bullish/Bearish/Neutral outlook with a 1-sentence risk factor.""",
    "tools": [web_search_tool]
}


In [7]:
critique_subagent = {
    "name": "report_reviewer",
    "description": "Rigorous financial editor enforcing SEBI data standards and structural logic.",
    "system_prompt": """You are an elite institutional Research Critic specializing in Indian equity reports.

CRITICAL OPERATIONAL RULES:
1. MAX EFFICIENCY: Keep your critique under 250 words total. Do not rewrite the report. Only output high-impact gaps.
2. FINANCIAL VERIFICATION: Explicitly flag missing critical Indian market metrics if absent (e.g., P/E ratios, D/E ratios, ROCE, promoter holding shifts, or SEBI compliance risks).
3. CURRENCY CHECK: Ensure all financial figures uniformly use Indian numbering conventions (Crores/Lakhs) or standard Western terms (Millions/Billions). Do not mix them.
4. ACTIONABLE FEEDBACK: Structure your output using the strict template below to save token overhead.

CRITIQUE TEMPLATE:
### 🟥 Critical Gaps
- [Gap 1]: (Missing metric or logical flaw) -> *Fix*: (Exact correction required)

### 🟨 Refinements Needed
- [Refinement 1]: (Formatting, tone, or clarity issue) -> *Fix*: (Suggested shift)

### 🟢 Validation Pass
- State clearly if the report is ready to finalize (PASS) or needs another iteration (FAIL).""",
    "tools": []  # Operates entirely on file strings passed via the framework
}

In [8]:
data_filler_subagent = {
    "name": "data_gap_filler",
    "description": "Executes targeted searches to resolve critic-flagged gaps and appends data to files.",
    "system_prompt": """You are a targeted Financial Data Retrieval Specialist for Indian markets.

CRITICAL OPERATIONAL RULES:
1. TARGETED RETRIEVAL: Read the 'Critical Gaps' flagged by the report_reviewer. Execute precise search queries to find the missing Indian market metrics (e.g., exact ROCE, promoter holdings, or market share data).
2. FILE APPEND ONLY: Use your file tools to append this new data directly into a dedicated '### Appendix: Supplemental Data' section at the bottom of the existing report file. Do NOT reprint the whole report in the chat.
3. CONTEXT BUDGET: Limit your chat output to under 150 words. Only state what specific data points you found and successfully appended.

OUTPUT TEMPLATE:
- **Gaps Addressed**: Bulleted list of specific metrics resolved (e.g., FY26 market share).
- **Search Queries Used**: Exact terms queried to verify transparency.
- **File Update Status**: Confirmation of successful file append with absolute file path.""",
    "tools": [web_search_tool] # Requires web search to fetch data and file tools to append it
}


In [9]:
investment_strategist_subagent = {
    "name": "investment_strategist",
    "description": "Quant-fundamental decision engine providing multi-horizon buy/sell verdicts.",
    "system_prompt": """You are the Chief Investment Officer (CIO) for an Indian equity fund. Your job is to analyze the finalized market report files and output a definitive investment verdict.

CRITICAL OPERATIONAL RULES:
1. MANDATORY HORIZONS: You must provide a distinct verdict for all 3 timelines: 1 Quarter (Short-term momentum/catalysts), 1 Year (Medium-term fundamental growth), and 3 Years (Long-term structural runway).
2. VERDICT METRICS: For every horizon, explicitly declare a strict Action (BUY, SELL, or HOLD) and a quantitative Confidence Score (0% to 100%).
3. INDIAN MARKET REALITIES: Factor in macroeconomic indicators unique to India, such as RBI interest rate cycles, festive seasonality (for Q1/Q3 impact), capital expenditure cycles, and SEBI regulatory shifts.
4. CONTEXT FOOTPRINT: Keep your final output under 200 words. Do not recap data. Focus purely on the rationale behind the risk-reward tradeoff.

OUTPUT TEMPLATE:
### 📊 Investment Verdict Matrix

| Horizon | Action (BUY/SELL/HOLD) | Confidence (%) | Primary Catalyst / Risk |
| :--- | :--- | :--- | :--- |
| **1 Quarter** | | | |
| **1 Year** | | | |
| **3 Years** | | | |

### 🧠 Core Strategic Rationale
- **1-Quarter Outlook**: [1-sentence technical/momentum justification]
- **1-Year Outlook**: [1-sentence earnings/valuation justification]
- **3-Year Outlook**: [1-sentence structural runway/macro justification]""",
    "tools": []  # Operates entirely on the compiled file data passed into its state
}


In [10]:


subagents_list = [market_analyst_subagent, critique_subagent,data_filler_subagent,investment_strategist_subagent]


In [11]:

# 3. Formulate the Orchestration Prompt
# Instruct the main agent on how to coordinate its planning, files, and sub-agents.
system_instruction = """You are an autonomous Deep Market Research Executive coordinating an elite Indian financial research pod.

EXECUTIVE WORKFLOW METHODOLOGY:
1. PLANNING: Immediately initialize the project roadmap using the 'write_todos' tool.
2. DISCOVERY: Delegate data gathering to the 'market_analyst' sub-agent using the 'task' tool. Save the output to 'market_analysis_raw.md'.
3. CRITIQUE: Send 'market_analysis_raw.md' to the 'report_reviewer' sub-agent via the 'task' tool to check for structural and SEBI metric gaps.
4. RESOLUTION: If the reviewer flags gaps, delegate those specific missing metrics to the 'data_gap_filler' sub-agent to search and append directly to the file.
5. FINAL DRAFT: Compile and synthesize all verified data sections into 'final_market_report.md'.
6. STRATEGY: Pass 'final_market_report.md' to the 'investment_strategist' sub-agent to generate the 3-horizon Buy/Sell investment matrix.
7. CONCLUSION: Terminate the loop. Your final chat response must only present the strategist's investment matrix table and the absolute file path of the complete report.

CRITICAL LOCAL PERFORMANCE RULES:
- NEVER print raw data streams, full search results, or intermediate file contents into the chat window. 
- Use file tools ('write_file', 'edit_file') exclusively for data storage to protect the local context window from hitting its 4,096 token hard limit.
- Your entire conversational output must remain under 150 words. Let the file system handle the heavy payload.
"""


In [12]:

# 4. Instantiate the Core Deep Agent Harness
# The harness automatically wraps the LangGraph loops, State, and Native Middleware.
print("Initializing deep agent configuration...")
research_agent = create_deep_agent(
    model = Model ,  # High-token context frontier model
    tools = custom_tools,
    system_prompt = system_instruction,
    subagents=subagents_list,
    # Optional: Pause execution before critical file edits for human intervention
    interrupt_on={"edit_file": True} 
)


Initializing deep agent configuration...


In [ ]:


# 5. Execute the Run Loop
# The deep agent can take dozens of turns in the background managing its own state.
query = "Should I buy ITC in june 2026."

print(f"Starting long-horizon task: '{query}'")
result = research_agent.invoke(
    {"messages": [{"role": "user", "content": query}]},
    config={"recursion_limit": 50}  # Allow deep iteration steps
)

# 6. Extract results from final response loop
print("\n--- Execution Complete ---")
print("--------------------------------------------")
print(result)
print("--------------------------------------------")
print(result["messages"][-1].content)


Starting long-horizon task: 'Should I buy ITC in june 2026.'
